In [1]:
import os
from dotenv import load_dotenv

load_dotenv()

NEO4J_URI = os.getenv("NEO4J_URI")
NEO4J_USERNAME = os.getenv("NEO4J_USERNAME")
NEO4J_PASSWORD = os.getenv("NEO4J_PASSWORD")
GROQ_API_KEY = os.getenv("GROQ_API_KEY")

In [2]:
from langchain_community.graphs import Neo4jGraph
graph = Neo4jGraph(
    url="neo4j+s://fa687069.databases.neo4j.io",
    username=NEO4J_USERNAME,
    password=NEO4J_PASSWORD,
    database="fa687069",
    refresh_schema=False
)
graph

In [3]:
movie_query = """
LOAD CSV WITH HEADERS FROM 'https://raw.githubusercontent.com/tomasonjo/blog-datasets/refs/heads/main/movies/movies_small.csv' AS row

MERGE (m:Movie {id: row.movieId})
SET m.released = date(row.released),
    m.title = row.title,
    m.imdbRating = toFloat(row.imdbRating)

WITH m, row
UNWIND split(row.director, '|') AS director
MERGE (d:Person {name: trim(director)})
MERGE (d)-[:DIRECTED]->(m)

WITH m, row
UNWIND split(row.actors, '|') AS actor
MERGE (a:Person {name: trim(actor)})
MERGE (a)-[:ACTED_IN]->(m)

WITH m, row
UNWIND split(row.genres, '|') AS genre
MERGE (g:Genre {name: trim(genre)})
MERGE (m)-[:IN_GENRE]->(g)
"""

In [4]:
graph._driver.session(database=graph._database).run(movie_query)

In [5]:
graph.refresh_schema()

In [6]:
graph.get_schema

'Node properties:\nMovie {id: STRING, released: DATE, title: STRING, imdbRating: FLOAT}\nPerson {name: STRING}\nGenre {name: STRING}\nRelationship properties:\n\nThe relationships:\n(:Movie)-[:IN_GENRE]->(:Genre)\n(:Person)-[:DIRECTED]->(:Movie)\n(:Person)-[:ACTED_IN]->(:Movie)'

In [7]:
from langchain_groq import ChatGroq

llm = ChatGroq(model='openai/gpt-oss-120b', groq_api_key=GROQ_API_KEY)

In [8]:
from langchain.chains import GraphCypherQAChain

chain = GraphCypherQAChain.from_llm(graph=graph, llm=llm, verbose=True)

chain

GraphCypherQAChain(verbose=True, graph=<langchain_community.graphs.neo4j_graph.Neo4jGraph object at 0x00000200787474F0>, cypher_generation_chain=LLMChain(prompt=PromptTemplate(input_variables=['question', 'schema'], template='Task:Generate Cypher statement to query a graph database.\nInstructions:\nUse only the provided relationship types and properties in the schema.\nDo not use any other relationship types or properties that are not provided.\nSchema:\n{schema}\nNote: Do not include any explanations or apologies in your responses.\nDo not respond to any questions that might ask anything else than for you to construct a Cypher statement.\nDo not include any text except the generated Cypher statement.\n\nThe question is:\n{question}'), llm=ChatGroq(client=<groq.resources.chat.completions.Completions object at 0x000002002BB40160>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x000002002BA7FC10>, model_name='openai/gpt-oss-120b', groq_api_key=SecretStr('******

In [11]:
response = chain.invoke({"query": "Who was the director of the movie Jumanji"})
response



> Entering new GraphCypherQAChain chain...
Generated Cypher:
MATCH (p:Person)-[:DIRECTED]->(m:Movie {title: "Jumanji"})
RETURN p.name AS director_name
Full Context:
[{'director_name': 'Joe Johnston'}]

> Finished chain.


{'query': 'Who was the director of the movie Jumanji',
 'result': 'Joe Johnston was the director of the movie Jumanji.'}

In [12]:
response = chain.invoke({"query": "Who was the actors of the movie Jumanji"})
response



> Entering new GraphCypherQAChain chain...
Generated Cypher:
MATCH (p:Person)-[:ACTED_IN]->(m:Movie {title: 'Jumanji'}) RETURN p.name
Full Context:
[{'p.name': 'Robin Williams'}, {'p.name': 'Bradley Pierce'}, {'p.name': 'Kirsten Dunst'}, {'p.name': 'Jonathan Hyde'}]

> Finished chain.


{'query': 'Who was the actors of the movie Jumanji',
 'result': 'Robin\u202fWilliams, Bradley\u202fPierce, Kirsten\u202fDunst, and Jonathan\u202fHyde were the actors of the movie\u202fJumanji.'}

In [13]:
response = chain.invoke({"query": "What is the genre of the movie Jumanji"})
response



> Entering new GraphCypherQAChain chain...
Generated Cypher:
MATCH (m:Movie {title: "Jumanji"})-[:IN_GENRE]->(g:Genre)
RETURN g.name
Full Context:
[{'g.name': 'Adventure'}, {'g.name': 'Children'}, {'g.name': 'Fantasy'}]

> Finished chain.


{'query': 'What is the genre of the movie Jumanji',
 'result': 'Adventure, Children, Fantasy.'}